# Phase 2 — CODI and KaVa on Kaggle

Runs the controlled GPT-2 comparison: both methods use `M=6` autoregressive continuous thoughts; KaVa adds compressed KV-trajectory supervision. Use a **P100** or **T4** GPU. The current trainer intentionally uses one GPU for deterministic checkpoint/resume.

Run CODI and KaVa as separate experiments. A full config is 96,405 optimizer steps, so expect multiple Kaggle sessions. Exit code 42 means the wall-clock guard saved a resumable checkpoint.

## 1. Clone a pinned code revision and install dependencies

In [ ]:
import os, subprocess

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
REPO_DIR = "/kaggle/working/latent-reasoning"
RUN_COMMIT = "main"  # For a full run, replace with the exact commit printed below.

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("PIN THIS COMMIT:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Optional: restore a previous session

Attach the previous notebook output as a Kaggle input, then copy the method directory back before training. Restore the whole directory so the checkpoint and `run_manifest.json` stay together. Skip on the first session.

In [ ]:
# Example — edit the attached-dataset path and choose one method:
# !mkdir -p outputs
# !cp -a /kaggle/input/<previous-output>/latent-reasoning/outputs/codi outputs/
# !cp -a /kaggle/input/<previous-output>/latent-reasoning/outputs/kava outputs/

## 3. Run all CPU tests

In [ ]:
!python -m pytest -q

## 4. Validate the controlled Phase-2 contract

Both reports must say `status: ok`, and `controlled_peer_differences` must be empty.

In [ ]:
!python scripts/validate_phase2.py --config configs/codi.yaml --peer-config configs/kava.yaml
!python scripts/validate_phase2.py --config configs/kava.yaml --peer-config configs/codi.yaml

## 5. One-step GPU smoke tests

Use disposable output directories. This verifies a real GPT-2 teacher target, continuous student forward, backward pass, checkpoint, and resume fingerprint before the long run.

In [ ]:
!python -m src.train.kaggle_run --config configs/codi.yaml --set output_dir=outputs/codi_smoke train.total_steps=1 train.ckpt_every=1
!python -m src.train.kaggle_run --config configs/kava.yaml --set output_dir=outputs/kava_smoke train.total_steps=1 train.ckpt_every=1

## 6. Train CODI

Re-run this exact cell after code 42. Do not change scientific settings or checkout a different commit while resuming.

In [ ]:
!python -u scripts/resume_training.py --config configs/codi.yaml

## 7. Evaluate CODI (quick gate, then full)

In [ ]:
!python -m src.eval.run_eval --config configs/codi.yaml --limit 200
# !python -m src.eval.run_eval --config configs/codi.yaml

## 8. Train and evaluate KaVa

Start only after the CODI run is safely persisted, unless you intentionally want to spend another set of sessions in parallel.

In [ ]:
!python -u scripts/resume_training.py --config configs/kava.yaml
!python -m src.eval.run_eval --config configs/kava.yaml --limit 200
# !python -m src.eval.run_eval --config configs/kava.yaml

## 9. Persist progress at the end of every session

Preferred: use **Save Version → Save & Run All / Quick Save with outputs enabled**, then attach that notebook output in the next session. Checkpoints are already atomic and each method keeps only the latest two.

For a manual download, archive only the newest checkpoint plus metadata and split it into browser-friendly pieces. Replace `codi` with `kava` as needed.

In [ ]:
METHOD = "codi"
from pathlib import Path
checkpoints = sorted((Path("outputs") / METHOD / "checkpoints").glob("step_*.pt"))
assert checkpoints, "No checkpoint found"
latest = checkpoints[-1]
archive = f"phase2_{METHOD}_latest.tar"
files = [str(latest), f"outputs/{METHOD}/run_manifest.json", f"outputs/{METHOD}/phase2_validation.json"]
subprocess.run(["tar", "-cf", archive, *files], check=True)
subprocess.run(["split", "-b", "1500m", "-d", "-a", "2", archive, archive + ".part-"], check=True)
print("Created:", sorted(str(path) for path in Path(".").glob(archive + ".part-*")))